# Accessing demands

This is a very practical example showing how to access and aggregate demand data from multiple buildings.

## Create a branch with a bunch of buildings

Let's start with a function to create a dummy electricity demand profile for a building:

In [1]:
import numpy as np
import pandas as pd

def create_electricity_demand_profile(timeindex:pd.DatetimeIndex, total:float) -> pd.Series:

    # Generate a dummy electricity demand profile (e.g., sinusoidal pattern)
    hours_in_day = 24
    days_in_year = 365
    demand_profile = 50 + 30 * np.sin(2 * np.pi * (timeindex.hour + timeindex.dayofyear * hours_in_day) / (hours_in_day * days_in_year))

    return pd.Series(data=demand_profile, index=timeindex)

In [3]:
from odeon.model import Branch, Building, Household, Commercial

branch = Branch(name="My Branch", year=2023)

building_names_n_households_n_commercials = {
    "Lakeview apartment complex": (5, 0),
    "Office": (0, 2),
    "Shopping mall": (0, 5),
    "Suburban home": (2, 0),
    "Mixed-use building": (6, 2),
}

for building_name, (n_households, n_commercials) in building_names_n_households_n_commercials.items():

    # create the building itself:
    building = Building(name=building_name)

    # create households and add demands:
    for i in range(n_households):

        # the household itself:
        household = Household(name=f"Household {i+1}")

        # create and add the electricity demand profile:
        demand_srs = create_electricity_demand_profile(branch.timeindex, total=5000)  # total annual consumption in kWh
        household.electricity_demand = demand_srs

        building.add_building_units(household)

    # create commercials and add demands:
    for j in range(n_commercials):
        commercial = Commercial(name=f"Commercial {j+1}")
        building.add_building_units(commercial)

    # add the building to the branch:
    branch.add_objects(building)
